In [1]:
# Cell 1: Setup & Constants
# Notebook 05: Dim_Policy — Gold_SalesOps_Dim_Policy
# Source: Policy (Silver)
# Grain: PolicyId (one row per policy — confirmed unique)

from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")

SILVER_BASE = "abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo"

print("Setup complete.")
print(f"Silver base: {SILVER_BASE}")

StatementMeta(, 1c090095-df24-4c30-a780-89199cfe4ea3, 3, Finished, Available, Finished, False)

Setup complete.
Silver base: abfss://CRBDATA_PAS_EM20_PRE@onelake.dfs.fabric.microsoft.com/PAS_U_EM20_LH_Silver.Lakehouse/Tables/dbo


In [2]:
# Cell 2: Load Policy table

df_policy = (
    spark.read.format("delta").load(f"{SILVER_BASE}/Policy")
    .filter(F.col("IsDeleted") == False)
)

total = df_policy.count()
distinct = df_policy.select("PolicyId").distinct().count()

print(f"Policy rows:        {total:,}")
print(f"Distinct PolicyId:  {distinct:,}")
print(f"PolicyId unique?    {'YES' if distinct == total else 'NO — DUPLICATES EXIST'}")

StatementMeta(, 1c090095-df24-4c30-a780-89199cfe4ea3, 4, Finished, Available, Finished, False)

Policy rows:        19,709,725
Distinct PolicyId:  19,709,725
PolicyId unique?    YES


In [3]:
# Cell 3: Select Final Columns

dim_policy = df_policy.select(
    # Key
    "PolicyId",
    "SourceId",
    "PolicyKey",
    # Dates
    "InceptionDate",
    "FirstInceptionDate",
    "ExpiryDate",
    "RenewalDate",
    # Policy Details
    "PolicyReference",
    "PolicyDescription",
    "RefPolicyStatusId",
    "RefPolicyStatus",
    "RefInsuranceTypeId",
    "RefInsuranceType",
    "OpportunityType",
    "IsRenewable",
    "IsWholeOrder",
    # Currency
    "GlobalCurrencyCode",
    # Ownership
    "OwnershipOrganisationId",
    "OwnershipOrganisation",
    # Financial Dimensions
    "GlobalFinancialGeographyId",
    "GlobalFinancialSegmentId",
    "GlobalLegalEntityId",
    # Renewal Linkage
    "RenewedFromPolicyId",
)

row_count = dim_policy.count()
print(f"Dim_Policy rows: {row_count:,}")

# NULL counts for key columns
print("\nKey column NULL counts:")
print("-" * 55)
for col_name in ["InceptionDate", "ExpiryDate", "RenewalDate", "RefPolicyStatus", "RefInsuranceType", "GlobalFinancialGeographyId", "OwnershipOrganisationId", "RenewedFromPolicyId"]:
    null_count = dim_policy.filter(F.col(col_name).isNull()).count()
    pct = null_count / row_count * 100 if row_count > 0 else 0
    print(f"  {col_name:<35} {null_count:>12,}  ({pct:5.1f}%)")

StatementMeta(, 1c090095-df24-4c30-a780-89199cfe4ea3, 5, Finished, Available, Finished, False)

Dim_Policy rows: 19,709,725

Key column NULL counts:
-------------------------------------------------------
  InceptionDate                             22,331  (  0.1%)
  ExpiryDate                                53,442  (  0.3%)
  RenewalDate                            7,216,207  ( 36.6%)
  RefPolicyStatus                            3,258  (  0.0%)
  RefInsuranceType                           3,244  (  0.0%)
  GlobalFinancialGeographyId               842,943  (  4.3%)
  OwnershipOrganisationId                  272,540  (  1.4%)
  RenewedFromPolicyId                    5,411,574  ( 27.5%)


In [4]:
# Cell 4: Write to Gold Lakehouse

dim_policy.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("Gold_SalesOps_Dim_Policy")

final_count = spark.read.table("Gold_SalesOps_Dim_Policy").count()
print(f"Gold_SalesOps_Dim_Policy written: {final_count:,} rows")

StatementMeta(, 1c090095-df24-4c30-a780-89199cfe4ea3, 6, Finished, Available, Finished, False)

Gold_SalesOps_Dim_Policy written: 19,709,725 rows
